In [ ]:
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [ ]:
import pandas as pd
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape); print(df.columns.tolist())

(7043, 21)
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [ ]:
df = df.drop(columns=["customerID"])
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df.isna().sum()[df.isna().sum() > 0])

TotalCharges    11
dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

# naye customers (tenure=0) ka total charge 0 hota hai
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# target ko numeric banao
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7   # apna number rakho
)

# raw test data + target -> Streamlit app ke liye
test_df = X_test.copy()
test_df["Churn"] = y_test
test_df.to_csv("test_data.csv", index=False)

print(X_train.shape, X_test.shape)
print(y.value_counts(normalize=True))

(5634, 19) (1409, 19)
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols + ["SeniorCitizen"]]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], remainder="passthrough")   # SeniorCitizen waise hi aage jayega

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=7),
    "KNN": KNeighborsClassifier(n_neighbors=15),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=7),
}

pipelines = {name: Pipeline([("prep", preprocess), ("clf", m)])
             for name, m in models.items()}
print(list(pipelines.keys()))

['Logistic Regression', 'Decision Tree', 'KNN', 'Naive Bayes', 'SVM']


In [ ]:
import os, joblib, sklearn
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

os.makedirs("model", exist_ok=True)
results = []

for name, m in models.items():
    pipe = Pipeline([("prep", clone(preprocess)), ("clf", m)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
    })

    fname = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipe, f"model/{fname}")

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df.to_csv("model/metrics.csv")
print(results_df)
print("sklearn version:", sklearn.__version__)

                     Accuracy     AUC  Precision  Recall      F1     MCC
Model                                                                   
Logistic Regression    0.8098  0.8573     0.6721  0.5535  0.6070  0.4871
Decision Tree          0.8013  0.8418     0.6416  0.5695  0.6034  0.4729
KNN                    0.7963  0.8351     0.6276  0.5722  0.5986  0.4634
Naive Bayes            0.7189  0.8306     0.4835  0.8636  0.6200  0.4690
SVM                    0.8091  0.8240     0.6966  0.4973  0.5803  0.4722
sklearn version: 1.6.1


In [ ]:
   !zip -r submission_files.zip model test_data.csv
   files.download("submission_files.zip")

  adding: model/ (stored 0%)
  adding: model/svm.joblib (deflated 91%)
  adding: model/naive_bayes.joblib (deflated 64%)
  adding: model/decision_tree.joblib (deflated 66%)
  adding: model/knn.joblib (deflated 91%)
  adding: model/metrics.csv (deflated 35%)
  adding: model/logistic_regression.joblib (deflated 69%)
  adding: test_data.csv (deflated 88%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>